# Trustworthy RAG — Kaggle GPU run
**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**. Add the private dataset containing `trag_bundle`.
Then *Save Version → Save & Run All (Commit)*: it runs in the background; download `trag_results_<STAGE>.zip` from the Output tab.
No API keys are needed or used here.

In [ ]:
# ---- settings ----
STAGE = "core"   # "core" = original plan (mid-sem base). "extensions" = evidence-removal + oracle conditions.
SYSTEMS = {"core": ["trust_rag", "vanilla_rag", "closed_book"],
           "extensions": ["removed_trust", "removed_vanilla", "oracle_trust", "oracle_vanilla"]}[STAGE]
RUN_RAGTRUTH = (STAGE == "core")
MODELS = [("qwen1.5b", "qwen2.5:1.5b", 11434, 0), ("llama8b", "llama3.1:8b", 11435, 1)]
import glob, os, shutil, subprocess, time, zipfile, pathlib
src = next(p for p in glob.glob("/kaggle/input/**/src", recursive=True) if os.path.isdir(p) and os.path.exists(os.path.join(os.path.dirname(p), "configs")))
BUNDLE = os.path.dirname(src); WORK = "/kaggle/working/trustworthy-rag"
if not os.path.exists(WORK):
    shutil.copytree(BUNDLE, WORK)
# resume: copy runs from any previously attached results
for prev in glob.glob("/kaggle/input/**/experiments/runs/main_*", recursive=True):
    dst = os.path.join(WORK, "experiments/runs", os.path.basename(prev))
    if not os.path.exists(dst): shutil.copytree(prev, dst); print("resumed", prev)
assert not os.path.exists(os.path.join(WORK, ".env")), ".env must never be uploaded"
import torch; NGPU = torch.cuda.device_count(); print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
import shutil, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd[:90]}\n{(r.stdout + r.stderr)[-600:]}")
    return r.returncode
sh('pip -q install "transformers==4.49.0" "sentence-transformers==5.7.0" rank_bm25 pyyaml')
# Ollama's installer needs zstd (absent from Kaggle's image); show its output; fall back to direct downloads
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils")
sh("curl -fsSL https://ollama.com/install.sh | sh")
if not shutil.which("ollama"):
    sh("curl -fsSL -o /tmp/ollama.tar.zst https://ollama.com/download/ollama-linux-amd64.tar.zst && tar --use-compress-program=unzstd -C /usr/local -xf /tmp/ollama.tar.zst")
if not shutil.which("ollama"):
    sh("curl -fsSL -o /tmp/ollama.tgz https://ollama.com/download/ollama-linux-amd64.tgz && tar -C /usr/local -xzf /tmp/ollama.tgz")
OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
assert os.path.exists(OLLAMA), "Ollama could not be installed - send Claude the output of this cell"
sh(f"{OLLAMA} --version")

In [ ]:
servers = []
for tag, model, port, gpu in MODELS:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % max(NGPU, 1)), OLLAMA_HOST=f"127.0.0.1:{port}",
               OLLAMA_NUM_PARALLEL="4", OLLAMA_FLASH_ATTENTION="1", OLLAMA_KEEP_ALIVE="24h")
    servers.append(subprocess.Popen([OLLAMA, "serve"], env=env, stdout=open(f"/kaggle/working/ollama_{port}.log", "w"), stderr=subprocess.STDOUT))
time.sleep(10)
for tag, model, port, gpu in MODELS:
    r = subprocess.run([OLLAMA, "pull", model], env=dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}"), capture_output=True, text=True)
    print(model, "pulled" if r.returncode == 0 else r.stderr[-500:])
# sanity check: every model must answer before the long run starts
import requests
for tag, model, port, gpu in MODELS:
    for _ in range(30):
        try:
            r = requests.post(f"http://127.0.0.1:{port}/api/chat", timeout=300, json={"model": model, "stream": False,
                              "messages": [{"role": "user", "content": "Reply with the word OK."}]})
            if r.ok: break
        except Exception as e:
            pass
        time.sleep(10)
    assert r.ok, f"{model} not answering: {r.text[:300]}"
    print(tag, "->", r.json()["message"]["content"][:40])

In [ ]:
# ---- generation: both models in parallel (one per GPU) ----
t0 = time.time()
gens = {tag: subprocess.Popen(["python", "-u", "-m", "scripts.run_generation", "--config", f"configs/kaggle_{tag}.yaml", "--systems", *SYSTEMS],
                              cwd=WORK, env=dict(os.environ, CUDA_VISIBLE_DEVICES=""), stdout=open(f"/kaggle/working/gen_{tag}.log", "w"), stderr=subprocess.STDOUT)
        for tag, *_ in MODELS}
while any(p.poll() is None for p in gens.values()):
    time.sleep(300)
    for tag in gens:
        tail = open(f"/kaggle/working/gen_{tag}.log").read().replace(chr(13), chr(10)).strip().splitlines()[-1:]
        print(f"[{(time.time()-t0)/60:.0f} min] {tag}: {tail}")
print({t: p.returncode for t, p in gens.items()})
for s in servers: s.terminate()   # free GPU memory for scoring

In [ ]:
# ---- scoring (NLI attribution + metrics), one model per GPU ----
sc = {tag: subprocess.Popen(["python", "-u", "-m", "scripts.score_run", "--run", f"main_{tag}"], cwd=WORK,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % max(NGPU, 1)), HF_HUB_DISABLE_SYMLINKS_WARNING="1"),
                            stdout=open(f"/kaggle/working/score_{tag}.log", "w"), stderr=subprocess.STDOUT)
      for tag, model, port, gpu in MODELS}
while any(p.poll() is None for p in sc.values()):
    time.sleep(300); print(f"[{(time.time()-t0)/60:.0f} min] scoring...", {t: p.poll() for t, p in sc.items()})
print({t: p.returncode for t, p in sc.items()})

In [ ]:
# ---- external benchmark: RAGTruth QA (core stage only) ----
if RUN_RAGTRUTH:
    r = subprocess.run(["python", "-u", "-m", "scripts.eval_ragtruth"], cwd=WORK, capture_output=True, text=True,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    print(r.stdout[-3000:]); print(r.stderr[-1500:] if r.returncode else "")

In [ ]:
# ---- package results ----
out = f"/kaggle/working/trag_results_{STAGE}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for tag, *_ in MODELS:
        for f in pathlib.Path(WORK, "experiments/runs", f"main_{tag}").glob("*"):
            z.write(f, f.relative_to(WORK))
    for f in pathlib.Path(WORK, "experiments/results").rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(WORK))
    for f in glob.glob("/kaggle/working/*.log"): z.write(f, "logs/" + os.path.basename(f))
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")